In [2]:
import pandas as pd
import numpy
import re
from datetime import datetime

In [5]:
# Function to extract year from sheet name
def extract_year(sheet_name):
    match = re.search(r"\b\d{4}\b", sheet_name)  # Finds a 4-digit year
    return int(match.group()) if match else None

def clean_municipality_name(name):
    replacements = {
        "r.": "rajono",
        "m.": "miesto",
        "sav.": "savivaldybe"
    }
    ltu_mapping = str.maketrans("ąčęėįšųūž", "aceeisuuz")

    cleaned = name.lower()

    for old, new in replacements.items():
        cleaned = cleaned.replace(old, new)

    cleaned = cleaned.translate(ltu_mapping)

    cleaned = cleaned.replace(" ", "-")

    cleaned = cleaned.strip("-")
    return cleaned

In [6]:
file_path = "./vmi/Paramos apskaiciavimo statistika 2024-11-18.xlsx"

# Load the Excel file
excel_file = pd.ExcelFile(file_path, engine="openpyxl")

# Filter sheet names that start with "Apskaičiuota" and contain a year
sheets_with_years = [(sheet, extract_year(sheet)) for sheet in excel_file.sheet_names if sheet.startswith("Apskaičiuota")]

columns = "A,B,C,F,I"
headers = ["municipality", "legal_id", "entity_name", "total_funders", "total_funds"]

# Read and combine all data into a single DataFrame
df_list = []

for sheet_name, year in sheets_with_years:
    temp_df = pd.read_excel(excel_file, sheet_name=sheet_name, header=None, skiprows=7, usecols=columns, names=headers)  # Start reading from 8th row
    temp_df["year"] = year  # Add the extracted year as a new column
    df_list.append(temp_df)

# Concatenate all DataFrames into one
final_df = pd.concat(df_list, ignore_index=True)

# Get the value from "Pagal_teisines_formas" sheet, cell A3
pagal_teisines_formas = pd.read_excel(excel_file, sheet_name="Pagal_teisines_formas", header=None)
value_from_A3 = pagal_teisines_formas.iloc[2, 0]  # A3 is [2,0] in 0-based indexing
record_dt = value_from_A3.split(":")[1].strip()

final_df["record_dt"] = record_dt

final_df["municipality"] = final_df["municipality"].apply(clean_municipality_name)


In [ ]:
summary_df = final_df.groupby(["municipality", "year"], as_index=False).agg({"total_funders": "sum", "total_funds": "sum", "legal_id": "count" })
summary_df = summary_df.rename(columns={"legal_id": "total_reciever"})

summary_df["record_dt"] = final_df["record_dt"]
summary_df

,municipality,year,total_funders,total_funds,total_reciever,record_dt
0,akmenes-rajono-savivaldybe,2019,2521,89021.80,157,2024-11-18
1,akmenes-rajono-savivaldybe,2020,2349,83157.54,152,2024-11-18
2,akmenes-rajono-savivaldybe,2021,2347,188606.10,147,2024-11-18
3,akmenes-rajono-savivaldybe,2022,2284,121012.23,133,2024-11-18
4,akmenes-rajono-savivaldybe,2023,2445,135669.70,133,2024-11-18
...,...,...,...,...,...,...
295,zarasu-rajono-savivaldybe,2019,2187,73935.27,160,2024-11-18
296,zarasu-rajono-savivaldybe,2020,2072,76644.54,160,2024-11-18
297,zarasu-rajono-savivaldybe,2021,2080,84269.40,153,2024-11-18
298,zarasu-rajono-savivaldybe,2022,1864,90064.48,144,2024-11-18
